In [0]:
# Section 1 — Read Bronze ERP Location Data

df = spark.table("bike_lakehouse.bronze.erp_loc_a101")

display(df)

In [0]:
# Section 2 — Basic Data Quality Profile

from pyspark.sql.functions import col

print("Total rows:", df.count())

for column_name in df.columns:
    print(
        column_name,
        "NULLs:",
        df.filter(col(column_name).isNull()).count()
    )

In [0]:

# Section 3 — Inspect Country Values

display(
    df
    .groupBy("CNTRY")
    .count()
    .orderBy(col("count").desc())
)

In [0]:

# Section 4 — Inspect Missing Country Records

display(
    df
    .filter(
        col("CNTRY").isNull() |
        (col("CNTRY").cast("string").rlike(r"^\s*$"))
    )
    .select("CID", "CNTRY")
)

In [0]:
# Section 5 — Standardize Country Values

from pyspark.sql.functions import trim, upper, when

df_clean = (
    df
    .withColumn(
        "CNTRY",
        when(
            upper(trim(col("CNTRY"))).isin("US", "USA", "UNITED STATES"),
            "United States"
        )
        .when(
            upper(trim(col("CNTRY"))).isin("DE", "GERMANY"),
            "Germany"
        )
        .when(
            trim(col("CNTRY")) == "",
            "Unknown"
        )
        .when(
            col("CNTRY").isNull(),
            "Unknown"
        )
        .otherwise(trim(col("CNTRY")))
    )
)

display(
    df_clean
    .groupBy("CNTRY")
    .count()
    .orderBy(col("count").desc())
)

In [0]:
# Section 6 — Check Customer ID Uniqueness

print("Total rows:", df_clean.count())

print(
    "Duplicate CID groups:",
    df_clean
    .groupBy("CID")
    .count()
    .filter(col("count") > 1)
    .count()
)

In [0]:
# Section 7 — Rename Columns

df_silver = (
    df_clean
    .withColumnRenamed("CID", "customer_id")
    .withColumnRenamed("CNTRY", "country")
)

df_silver.printSchema()

In [0]:
# Section 8 — Final Silver Sanity Check

print("Final row count:", df_silver.count())

print(
    "Duplicate customer IDs:",
    df_silver
    .groupBy("customer_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

print(
    "NULL customer IDs:",
    df_silver.filter(col("customer_id").isNull()).count()
)

print(
    "NULL countries:",
    df_silver.filter(col("country").isNull()).count()
)

display(
    df_silver
    .groupBy("country")
    .count()
    .orderBy(col("count").desc())
)

In [0]:
# Section 9 — Write Silver Table

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bike_lakehouse.silver.erp_locations")